# Retraction Contamination Propagation Analysis
Exploratory analysis of how retracted papers' influence propagates through citation networks.

**Research Question**: How deep and wide does the influence of retracted papers spread, and how effective is scientific self-correction?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("../data/processed")

In [ ]:
retracted_df = pd.read_parquet(DATA_DIR / "retracted_papers.parquet")
citers_df = pd.read_parquet(DATA_DIR / "retraction_citers.parquet")

print(f"Retracted papers: {len(retracted_df)}")
print(f"Propagation records: {len(citers_df)}")
print(f"Unique retracted papers with citers: {citers_df['retracted_doi'].nunique()}")

## 1. Retracted Papers Overview


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(retracted_df["cited_by_count"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Citation Count")
axes[0].set_ylabel("Number of Retracted Papers")
axes[0].set_title("Citation Distribution of Retracted Papers")
axes[0].set_yscale("log")

retracted_df["publication_year"].dropna().astype(int).hist(
    ax=axes[1], bins=30, edgecolor="black", alpha=0.7
)
axes[1].set_xlabel("Publication Year")
axes[1].set_ylabel("Number of Retracted Papers")
axes[1].set_title("Retracted Papers by Publication Year")

plt.tight_layout()
plt.show()

## 2. Citation Propagation Scale


In [ ]:
citers_per_paper = (
    citers_df.groupby("retracted_doi").size().reset_index(name="num_citers")
)
citers_per_paper = citers_per_paper.merge(
    retracted_df[["doi", "title", "cited_by_count"]],
    left_on="retracted_doi",
    right_on="doi",
    how="left",
)

fig, ax = plt.subplots(figsize=(14, 8))
top20 = citers_per_paper.nlargest(20, "num_citers")
ax.barh(range(len(top20)), top20["num_citers"])
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(
    [t[:60] + "..." if len(str(t)) > 60 else str(t) for t in top20["title"]], fontsize=9
)
ax.set_xlabel("Number of Citing Papers (depth=1)")
ax.set_title("Top 20 Most-Cited Retracted Papers")
plt.tight_layout()
plt.show()

## 3. Temporal Pattern: Are Citations Declining After Retraction?


In [ ]:
# For each retracted paper, compare citing year vs retraction year
merged = citers_df.merge(
    retracted_df[["doi", "publication_year"]].rename(
        columns={"publication_year": "retraction_pub_year"}
    ),
    left_on="retracted_doi",
    right_on="doi",
    how="left",
)
merged["years_after_publication"] = (
    merged["citing_year"] - merged["retraction_pub_year"]
)

fig, ax = plt.subplots(figsize=(14, 6))
merged["years_after_publication"].dropna().hist(
    bins=range(-5, 30), ax=ax, edgecolor="black", alpha=0.7
)
ax.axvline(x=0, color="red", linestyle="--", label="Publication Year")
ax.set_xlabel("Years After Retracted Paper Publication")
ax.set_ylabel("Number of Citations")
ax.set_title("When Do Citations to Retracted Papers Occur?")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Summary Statistics


In [ ]:
print("=== Retraction Propagation Summary ===")
print(f"Total retracted papers analyzed: {retracted_df.shape[0]}")
print(
    f"Papers with OpenAlex data: {len(retracted_df[retracted_df['cited_by_count'] > 0])}"
)
print(f"Total direct citations collected: {len(citers_df)}")
print(
    f"Mean citations per retracted paper: {citers_df.groupby('retracted_doi').size().mean():.1f}"
)
print(
    f"Median citations per retracted paper: {citers_df.groupby('retracted_doi').size().median():.1f}"
)
print(f"Max citations: {citers_df.groupby('retracted_doi').size().max()}")

print(
    f"\nCiting papers year range: {int(citers_df['citing_year'].min())}-{int(citers_df['citing_year'].max())}"
)
print(
    f"Average citing paper citations: {citers_df['citing_cited_by_count'].mean():.1f}"
)

## Next Steps
- 2-hop and 3-hop propagation tracing using `RetractionPropagationAnalyzer`
- Citation context analysis using `CitationContextClassifier`
- Self-correction analysis: citation rate change post-retraction
- Cross-field comparison of contamination patterns
